# 05 · Valor de negocio

Traduce las alertas a energía y plata. Responde tres preguntas: cuánto se pierde en cada alerta, cuánto
se ahorra frente a la revisión mensual de hoy y cada cuántos días conviene limpiar.

**Parámetros editables** (están en `src/negocio.py`, con su fuente):

| Parámetro | Valor | Fuente |
|---|---|---|
| Precio de la energía | USD 42,84/MWh | Precio promedio adjudicado a la solar en RenovAr Ronda 2 ([Energía Estratégica, 23/01/2018](https://energiaestrategica.com/energiaestrategica/en/notes/solo-dos-empresas-ganadoras-la-licitacion-energias-renovables-argentinas)) |
| Costo de limpieza | USD 800 por MW | **SUPUESTO** (sin fuente argentina). Techo de contexto: O&M total de NREL de USD 9,1/kW/año ([Reuters Events](https://www.reutersevents.com/renewables/node/60975)) |
| Revisión manual | cada 30 días | **SUPUESTO**: hoy alguien mira los números una vez por mes |

> **Sobre el precio.** Las plantas son chinas (Hebei, 2018-2019). El precio de RenovAr Ronda 2 (2017) se usa **como ilustración del valor en un contexto argentino**, no como el ingreso real de esas plantas, que venden con otra tarifa y en otra moneda. Los USD de este notebook responden "¿cuánto valdría esto en un parque argentino?".

Requiere haber corrido `03_modelos` y `04_deteccion` (generan `models/`).

In [ ]:
# Si corrés en Colab: clonar el repo (una vez por sesión), pararse en él e instalar dependencias
import os, sys
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/tpo-solar'):
        !git clone https://github.com/imelinc/tpo-solar.git /content/tpo-solar
    %cd /content/tpo-solar
    !pip install -q -r requirements.txt
sys.path.append('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
from src.config import DATA_RAW, DATA_PROCESSED, MODELS

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.negocio import (PRECIO_USD_MWH, COSTO_LIMPIEZA_USD_MW, DIAS_REVISION_MANUAL,
                         t_optimo_limpieza, costo_anual_limpieza, ahorro_anual_limpieza,
                         T_BASE_LIMPIEZA_DIAS)
from src.deteccion import diario_flota, semaforo_flota
from src.features import agregar_dias_desde_lluvia

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 200)
# Editables (por defecto, los de src/negocio.py)
PRECIO = PRECIO_USD_MWH          # USD/MWh
COSTO_LIMPIEZA = COSTO_LIMPIEZA_USD_MW   # USD por MW por limpieza (SUPUESTO)
DIAS_REVISION = DIAS_REVISION_MANUAL     # días (SUPUESTO)
E = diario_flota("2018-06-30", "2019-06-13")
REFS = sorted(E.loc[E["modelo"] == "propio", "plant_id"].unique())
print(f"Días-planta: {len(E)} | precio USD {PRECIO}/MWh | limpieza USD {COSTO_LIMPIEZA}/MW (supuesto)")

## 1. Energía y plata perdidas por alerta

Un **episodio** son días seguidos con alerta del detector elegido en `04` en la misma planta. La pérdida es lo esperado con la planta limpia menos lo real, en los días del episodio.

In [ ]:
E = E.sort_values(["plant_id", "date_local"])
E["fecha"] = pd.to_datetime(E["date_local"])
nuevo = E["alerta_elegida"] & ~E.groupby("plant_id")["alerta_elegida"].shift(fill_value=False)
E["episodio"] = nuevo.groupby(E["plant_id"]).cumsum().where(E["alerta_elegida"])
ep = (E.dropna(subset=["episodio"]).groupby(["plant_id", "episodio"])
        .agg(inicio=("date_local", "first"), dias=("date_local", "size"), mwh=("perdida_mwh", "sum"),
             tipo=("tipo_alerta", lambda s: s[s != ""].mode().iloc[0] if (s != "").any() else "propio"))
        .reset_index())
ep["usd"] = ep["mwh"] * PRECIO
ep["usd_dia"] = ep["usd"] / ep["dias"]
print(f"Episodios: {len(ep)} | MWh perdidos en alertas: {ep['mwh'].sum():,.0f} | USD {ep['usd'].sum():,.0f}")
print(ep.groupby("plant_id").agg(episodios=("episodio", "size"), dias=("dias", "sum"), mwh=("mwh", "sum"), usd=("usd", "sum")).round(0))
print("\nLos 10 episodios más caros:")
print(ep.sort_values("usd", ascending=False).head(10)[["plant_id", "inicio", "dias", "tipo", "mwh", "usd", "usd_dia"]].round(1).to_string(index=False))

**Conclusión:** En todo el período hay 63 episodios de alerta, con 5.501 MWh perdidos (USD 235.684 a USD 42,84/MWh). **station09 explica el 86% de esa energía**: 4.724 MWh en 40 episodios y 114 días, y sus episodios más caros cuestan entre USD 1.470 y USD 2.505 por día. En el resto de las plantas los episodios duran 1 día (apagones) y suman entre USD 644 y USD 7.293 por planta. Los 2 episodios de station04 se miden contra la planta típica y su ficha está mal: no conviene contarlos.

## 2. Ahorro frente a la revisión mensual

Con la revisión mensual, una falla que aparece un día cualquiera se descubre en promedio a mitad del
ciclo (15 días) y se repara al día siguiente. Con el detector, se repara al día siguiente de la alerta.
Las demoras salen de las fallas sintéticas de `04` (mayo y junio, detector elegido). Si el detector no
encuentra la falla, se asume que la revisión mensual la encuentra igual.

In [ ]:
ev = pd.read_csv(MODELS / "evaluacion_detector.csv")
E_MW = E.loc[E["plant_id"].isin(REFS), "e_esp"].mean()      # MWh por día por MW (lo esperado, planta limpia)
print(f"Energía esperada media: {E_MW:.2f} MWh por día por MW")
filas = []
for _, r in ev.iterrows():
    if r["tipo"] == "abrupta":
        frac = float(str(r["caida"]).rstrip("%")) / 100
        nombre = f"Caída abrupta {r['caida']}"
    else:
        tasa = float(r["caida"])
        frac = tasa / 100 * 30 / 2     # pérdida media de 30 días de suciedad lineal
        nombre = f"Suciedad {tasa:.1f} %/día"
    det = r["detectadas_%"] / 100
    dcon = det * (np.nan_to_num(r["demora_mediana_dias"], nan=DIAS_REVISION / 2) + 1) + (1 - det) * (DIAS_REVISION / 2 + 1)
    dsin = DIAS_REVISION / 2 + 1
    filas.append({"falla": nombre, "perdida_mwh_dia_mw": frac * E_MW, "detectadas_%": r["detectadas_%"],
                  "dias_sin_atender_revision": dsin, "dias_sin_atender_detector": dcon,
                  "ahorro_mwh_mw": frac * E_MW * (dsin - dcon), "ahorro_usd_mw": frac * E_MW * (dsin - dcon) * PRECIO})
ahorro = pd.DataFrame(filas)
print(ahorro.round(2).to_string(index=False))
fig, ax = plt.subplots(figsize=(10, 3.8))
sns.barplot(data=ahorro, y="falla", x="ahorro_usd_mw", color="#31a354", ax=ax)
ax.set(xlabel="USD ahorrados por MW, por falla", ylabel="", title=f"Ahorro por falla frente a revisar cada {DIAS_REVISION} días (USD {PRECIO}/MWh)")
plt.show()

**Conclusión:** Con el detector, una caída de 50% se atiende en 1,4 días en lugar de 16, y se ahorran **USD 1.099 por MW por falla**. Un apagón total ahorra USD 2.141 por MW y una caída de 25%, USD 437. Las caídas de 10% casi no se detectan (USD 41 por MW). En la suciedad el ahorro es nulo o negativo: el detector la ve tarde o no la ve, y además la comparación con la revisión mensual no se ajusta bien a una pérdida que crece de a poco. Supuestos: revisión cada 30 días y reparación al día siguiente de la alerta.

## 3. Cada cuántos días conviene limpiar

$T^* = \sqrt{2C / (P \cdot E \cdot r)}$, con $C$ el costo de limpiar 1 MW, $P$ el precio, $E$ la energía
diaria por MW y $r$ la **tasa de ensuciamiento medida en los datos**: cuánto cae el índice real/esperado
(con la planta limpia) por cada día sin lluvia ≥ 5 mm. Como en Hebei la época seca es el invierno, y en
invierno también hay otras pérdidas (sombras, nieve), la tasa se mide **dentro de cada planta y mes**
(se resta la media de cada planta-mes) para no confundir suciedad con estación.

In [ ]:
A = pd.read_parquet(DATA_PROCESSED / "dataset_A_pvod_hourly.parquet")
lluvia = agregar_dias_desde_lluvia(A[["plant_id", "date_local", "om_precipitacion_mm"]].assign(date_local=A["date_local"].astype(str)))
lluvia = lluvia.drop_duplicates(["plant_id", "date_local"])
X = E[E["plant_id"].isin(REFS)].merge(lluvia[["plant_id", "date_local", "dias_desde_lluvia"]], on=["plant_id", "date_local"])
X = X[X["indice"].notna() & ~X["alerta_grave"] & X["dias_desde_lluvia"].between(1, 59)].copy()
X["mes"] = X["fecha"].dt.month
pend_bruta = np.polyfit(X["dias_desde_lluvia"], X["indice"], 1)[0]
g = X.groupby(["plant_id", "mes"])
xd = X["dias_desde_lluvia"] - g["dias_desde_lluvia"].transform("mean")
yd = X["indice"] - g["indice"].transform("mean")
coef, cov_w = np.polyfit(xd, yd, 1, cov=True)
pend, ee = coef[0], np.sqrt(cov_w[0, 0])
print(f"Pendiente bruta: {pend_bruta*100:.3f} %/día | dentro de planta y mes: {pend*100:.3f} %/día (error estándar {ee*100:.3f})")
por_planta = {p: np.polyfit(xd[X.plant_id == p], yd[X.plant_id == p], 1)[0] * 100 for p in REFS if (X.plant_id == p).sum() > 30}
print("Pendiente por planta (%/día, dentro de mes):", {k: round(v, 3) for k, v in por_planta.items()})
TASA = max(-pend, 0.0)
significativa = -pend > 2 * ee
print(f"Tasa de ensuciamiento usada: {TASA*100:.3f} %/día ({'distinta de 0 con 2 errores estándar' if significativa else 'NO distinguible de 0'})")
fig, ax = plt.subplots(figsize=(8, 3.8))
b = X.assign(bin=pd.cut(X["dias_desde_lluvia"], [0, 3, 7, 14, 21, 30, 45, 60]))
b.assign(yd=yd).groupby("bin", observed=True)["yd"].median().plot(marker="o", ax=ax)
ax.axhline(0, color="k", lw=0.8)
ax.set(xlabel="días desde la última lluvia ≥ 5 mm", ylabel="índice − media de la planta-mes",
       title="¿Cae el rendimiento a medida que pasan días sin lluvia?"); plt.show()

In [ ]:
if TASA > 0:
    T_OPT = t_optimo_limpieza(COSTO_LIMPIEZA, PRECIO, E_MW, TASA)
    t = np.arange(5, 366)
    costo = costo_anual_limpieza(t, COSTO_LIMPIEZA, PRECIO, E_MW, TASA)
    fig, ax = plt.subplots(figsize=(8, 3.8))
    ax.plot(t, costo); ax.axvline(T_OPT, color="red", ls="--", label=f"T* = {T_OPT:.0f} días")
    ax.set(xlabel="días entre limpiezas", ylabel="USD por MW por año", title="Costo anual: limpiezas + energía perdida por suciedad")
    ax.legend(); plt.show()
    print(f"T* = {T_OPT:.0f} días | costo anual en el óptimo: USD {costo_anual_limpieza(T_OPT, COSTO_LIMPIEZA, PRECIO, E_MW, TASA):,.0f} por MW")
else:
    T_OPT = np.inf
    print("Con esta tasa, limpiar no se paga: T* infinito (la lluvia alcanza).")
# Sensibilidad: el costo y dos de las tres tasas son SUPUESTOS. Ahorro = frente a limpiar una vez por año (también supuesto).
sens = pd.DataFrame([{"costo_usd_mw": cst, "tasa_pct_dia": r * 100, "T*_dias": t_optimo_limpieza(cst, PRECIO, E_MW, r),
                      "ahorro_usd_mw_anio": ahorro_anual_limpieza(cst, PRECIO, E_MW, r)}
                     for cst in [400, 800, 1600] for r in [0.0003, 0.00049, 0.0008]])
sens["origen"] = np.where(np.isclose(sens["tasa_pct_dia"], 0.049) & (sens["costo_usd_mw"] == 800),
                          "tasa medida + costo supuesto", "supuesto")
print(f"\nSensibilidad de T* y del ahorro anual por MW (vs. limpiar cada {T_BASE_LIMPIEZA_DIAS} días):")
print(sens.round({"T*_dias": 0, "ahorro_usd_mw_anio": 0}).to_string(index=False))


**Qué es dato y qué es supuesto en la tabla de sensibilidad:**

| Parámetro | Valores | Origen |
|---|---|---|
| Tasa de ensuciamiento | 0,049 %/día | **Medida** en los datos (dentro de planta y mes, sección 3) |
| Tasa de ensuciamiento | 0,03 y 0,08 %/día | **SUPUESTO**: escenarios bajo y alto alrededor de la medida |
| Costo de limpieza | USD 400, 800 y 1.600 por MW | **SUPUESTO** (800 es el valor base, sin cotización argentina) |
| Línea base del ahorro | limpiar una vez por año | **SUPUESTO** |
| Precio | USD 42,84/MWh | RenovAr Ronda 2 (ilustrativo, ver arriba) |


**Conclusión:** La tasa de ensuciamiento medida es de **0,049%/día** (error estándar 0,013), calculada dentro de cada planta y mes. Sin ese control sale casi 0, porque en Hebei la época seca coincide con el invierno. Varía mucho entre plantas (station06 −0,38%/día, station07 +0,24%/día), así que el promedio es solo indicativo. Con USD 800/MW por limpieza (supuesto) y USD 42,84/MWh, **T* = 148 días**: unas 2 o 3 limpiezas por año, además de las lluvias, con un costo anual de USD 3.956 por MW. La tabla de sensibilidad muestra que T* va de 33 a 206 días según el costo y la tasa: **el dato que más falta es una cotización real de limpieza**.

**Sensibilidad pedida** (costos USD 400/800/1.600 por MW, todos supuestos; tasas 0,03 y 0,08 %/día supuestas, 0,049 medida): T* va de **82 días** (USD 400, 0,08 %/día) a **266 días** (USD 1.600, 0,03 %/día). El ahorro anual frente a limpiar una vez por año va de **USD 219 a 4.829 por MW**; en el caso base (USD 800, tasa medida) es de **USD 1.743 por MW por año**, con T* = 147 días (148 con la tasa sin redondear).

## 4. Semáforo de la flota

In [ ]:
for fecha in ["2019-01-15", "2019-05-10"]:
    print(f"\nSemáforo al {fecha}:")
    print(semaforo_flota(fecha, PRECIO).round(2).to_string(index=False))

**Conclusión:** Al 15/1/2019, station02 y station09 están en rojo, y station06 en amarillo con aviso de sensor caído: en invierno su piranómetro lee menos del 30% de lo que da el satélite, algo a revisar en campo (puede ser nieve o suciedad sobre el sensor). Ese amarillo de station06 cae en enero-marzo y se marca como **baja confianza** (pérdida invernal que el modelo no aprende). station04 sale en **gris** (capacidad no confiable) en las dos fechas. Al 10/5/2019 solo station09 sigue en rojo. La pérdida en USD por día ayuda a priorizar: station09 pierde unos USD 1.600-2.300 por día, y el resto de las plantas, decenas o cientos.

## Resumen

**Para la gerencia:**
- La alerta diaria vale para fallas grandes: una caída de 50% o un apagón se atienden en 1-2 días en lugar de unos 16, y se ahorran entre USD 1.100 y 2.100 por MW por falla.
- La planta que más plata pierde es la que rinde mal todo el tiempo (station09: unos USD 200 mil en alertas durante el período). Una planta así no se ve si se compara solo contra el mes anterior.
- La suciedad se maneja con limpieza programada: con los supuestos actuales conviene limpiar cada unos 148 días. El resultado depende mucho del costo de limpieza, que todavía es un supuesto.
- Precio: USD 42,84/MWh (RenovAr Ronda 2, solar, 2017). Se aplica a plantas chinas **como ilustración** del valor en un contexto argentino, no es su ingreso real. Para un parque con otro contrato, cambiar `PRECIO` arriba.